# Council v3: six-attempt engineering screen on Kaggle T4 x2
This is not Test4Py upstream or proof of research improvement. No automatic full run.
Upload the supplied **test4py-screen-source.zip** as a Kaggle input, or use an updated
checkout. All six planned outcomes and failures are retained. Model quality may fail.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys, zipfile
from datetime import datetime, timezone
WORKDIR = Path('/kaggle/working/Test4Py_Setup_screen_v3')
# A supplied source bundle is preferable to an older remote checkout.
if not (WORKDIR / 'council_multitarget.py').exists():
    bundles = list(Path('/kaggle/input').rglob('test4py-screen-source.zip'))
    if len(bundles) == 1:
        WORKDIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(bundles[0]) as archive:
            for name in archive.namelist():
                if not (WORKDIR / name).resolve().is_relative_to(WORKDIR.resolve()):
                    raise ValueError('Unsafe archive member')
            archive.extractall(WORKDIR)
    else:
        # Kaggle may expand an uploaded dataset ZIP automatically.
        import shutil
        candidates = [p.parent for p in Path('/kaggle/input').rglob('council_multitarget.py')
                      if 'council-development-screen-v3' in p.read_text()]
        if len(candidates) != 1:
            raise RuntimeError('Add the supplied source ZIP/folder as Kaggle input, or set WORKDIR to updated source.')
        shutil.copytree(candidates[0], WORKDIR, dirs_exist_ok=True)
os.chdir(WORKDIR)
assert 'council-development-screen-v3' in (WORKDIR/'council_multitarget.py').read_text()
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
PREP = Path('/kaggle/working') / ('SCREEN_PREFLIGHT_' + STAMP)
OUTPUT = Path('/kaggle/working') / ('SCREEN_V3_' + STAMP)
print('Source:', WORKDIR, 'Output:', OUTPUT)


In [ ]:
# Prepare dependencies BEFORE the experiment; keep Kaggle CUDA Torch.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.6.17'], check=True)
subprocess.run([sys.executable, 'bootstrap_modern.py'], cwd=WORKDIR, check=True)
PYTHON = '/tmp/test4py-modern/bin/python'
subprocess.run([PYTHON, '-m', 'pip', 'install', 'bitsandbytes>=0.43,<1', 'accelerate>=0.29,<2'], check=True)
subprocess.run([PYTHON, '-c', "import torch; print(torch.__version__); print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]); assert torch.cuda.device_count() == 2"], check=True)


In [ ]:
# Real subprocess regression tests: no network/model inference in these tests.
subprocess.run([PYTHON, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_multitarget.py', '-v'], cwd=WORKDIR, check=True)
subprocess.run([PYTHON, 'council_multitarget.py', '--prepare-only', '--output', str(PREP)], cwd=WORKDIR, check=True)
preflight = json.loads((PREP/'preflight.json').read_text())
for name, row in preflight.items():
    print(name, 'ready=', row['ready'], 'commit=', row.get('commit'), 'error=', row.get('error'))
assert all(row['ready'] for row in preflight.values())


The next cell runs the fixed six-attempt GPU screen. Preparation has no model
inference. Proceed with the fixed plan; never choose targets or seeds based on
scores. Preserve this output even if readiness fails.


In [ ]:
cmd = [PYTHON, 'council_multitarget.py', '--prepared', str(PREP/'preflight.json'),
       '--targets', 'docstring_parser_parse,codetiming_timer', '--arms', 'baseline,ranked,verified',
       '--samples', '1', '--seed', '42', '--temperature', '0.7', '--max-new-tokens', '2048',
       '--coder-model', 'Qwen/Qwen2.5-Coder-7B-Instruct',
       '--critic-model', 'Qwen/Qwen3-4B-Instruct-2507', '--output', str(OUTPUT)]
run = subprocess.run(cmd, cwd=WORKDIR)
print('Runner exit:', run.returncode, '(2 means readiness failed; inspect retained artifacts)')


In [ ]:
from IPython.display import display, Markdown, FileLink
if (OUTPUT/'FINAL_REPORT.md').exists():
    display(Markdown((OUTPUT/'FINAL_REPORT.md').read_text()))
if (OUTPUT/'readiness.json').exists():
    gate = json.loads((OUTPUT/'readiness.json').read_text())
    print(json.dumps(gate, indent=2))
archive_path = Path(str(OUTPUT)+'.zip')
if archive_path.exists():
    display(FileLink(str(archive_path)))
assert run.returncode == 0, 'Engineering gate failed. Review archive; do not start full experiments.'
print('Automated gate passed. Assertion/target-reach review is still required. No full run launched.')
